In [ ]:
# 1) Mount Drive (if not yet)
# -------------------------
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, shutil, datetime, itertools
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import pandas as pd
from pathlib import Path
from google.colab import drive

In [6]:
# PATH SETUP
# =========================
DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks"
SPLIT80_DIR = f"{DRIVE_ROOT}/emotion/emotion_preprocessed_split80_20_aug4000"  # train/val

TEST_DIR = f"{DRIVE_ROOT}/emotion/emotion_preprocessed/test"

LOCAL_TRAIN = "/content/data/train"
LOCAL_VAL   = "/content/data/val"
LOCAL_TEST  = "/content/data/test"

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = f"{DRIVE_ROOT}/Emotion_VGG16_80_20_Improved_Results_ab{ts}"
os.makedirs(OUT_DIR, exist_ok=True)

def fresh_copy(src, dst):
    if os.path.exists(dst):
        shutil.rmtree(dst)
    # Use Pathlib to handle path concatenation for robustness
    shutil.copytree(Path(src), Path(dst))

print("Copying train/val/test to Colab local...")
fresh_copy(os.path.join(SPLIT80_DIR, "train"), LOCAL_TRAIN)
fresh_copy(os.path.join(SPLIT80_DIR, "val"),   LOCAL_VAL) # Fix: Ensure the correct path for validation data is used.
fresh_copy(TEST_DIR, LOCAL_TEST)
print("✅ Copy complete.")

Copying train/val/test to Colab local...
✅ Copy complete.


In [7]:
# ---------------- Data pipeline ----------------
IMG_SIZE_RAW = (48, 48)    # dataset images (grayscale)
IMG_SIZE_INP = (224, 224)  # model input
BATCH = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE


def make_loader(dir_path, shuffle=True, batch=BATCH):
    ds = image_dataset_from_directory(
        dir_path,
        labels="inferred",
        label_mode="categorical",   # one-hot for Keras metrics/ROC
        color_mode="grayscale",
        image_size=IMG_SIZE_RAW,
        batch_size=batch,
        shuffle=shuffle,
        seed=SEED
    )
    class_names = ds.class_names

    def _prep(x, y):
        x = tf.image.grayscale_to_rgb(x)
        x = tf.image.resize(x, IMG_SIZE_INP)
        x = tf.cast(x, tf.float32) / 255.0  # keep [0,1]; branch-wise preprocess later
        return x, y

    return ds.map(_prep, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE), class_names

train_ds, class_names = make_loader(LOCAL_TRAIN, shuffle=True)
val_ds, _             = make_loader(LOCAL_VAL,   shuffle=False)
test_ds, _            = make_loader(LOCAL_TEST,  shuffle=False)
num_classes = len(class_names)
print("Classes:", class_names, "| Num classes:", num_classes)


Found 28000 files belonging to 7 classes.
Found 5744 files belonging to 7 classes.
Found 7178 files belonging to 7 classes.
Classes: ['angry', 'disgusted', 'fearful', 'happy', 'neutral', 'sad', 'surprised'] | Num classes: 7


In [8]:

# ---------------- Hybrid model: VGG16 + CNN ----------------
from tensorflow.keras.applications import VGG16

# Backbones (frozen)
vgg_base = VGG16(include_top=False, weights="imagenet", input_shape=(224,224,3))
vgg_base.trainable = False

# Custom CNN model
def create_cnn_model(input_shape=(224,224,3)):
    model = models.Sequential()
    model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Conv2D(128, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Flatten())
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dropout(0.5))
    return model

cnn_base = create_cnn_model()

inp = layers.Input(shape=(224,224,3), name="inp_rgb01")  # [0,1] RGB

# Branch-1: VGG16 preprocess & features
x_v = layers.Lambda(lambda z: z * 255.0)(inp)
f_v = vgg_base(x_v, training=False)
f_v = layers.GlobalAveragePooling2D(name="gap_vgg16")(f_v)

# Branch-2: Custom CNN features
f_cnn = cnn_base(inp)

# Fusion + classifier head
f = layers.Concatenate(name="concat_feats")([f_v, f_cnn])
f = layers.BatchNormalization()(f)
f = layers.Dense(512, activation="relu")(f)
f = layers.BatchNormalization()(f)
f = layers.Dropout(0.5)(f)
out = layers.Dense(num_classes, activation="softmax", name="pred")(f)

model = models.Model(inp, out, name="Hybrid_VGG16_CNN")
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "Hybrid_VGG16_CNN"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ inp_rgb01           │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 224, 224,  │          0 │ inp_rgb01[0][0]   │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg16 (Functional)  │ (None, 7, 7, 512) │ 14,714,688 │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gap_vgg16           │ (None, 512)       │          0 │ vgg16[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 256)       │ 22,244,672 │ inp_rgb01[0][0]   │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_feats        │ (None, 768)       │          0 │ gap_vgg16[0][0],  │
│ (Concatenate)       │                   │            │ sequential[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 768)       │      3,072 │ concat_feats[0][… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 512)       │    393,728 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512)       │      2,048 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 512)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pred (Dense)        │ (None, 7)         │      3,591 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 37,361,799 (142.52 MB)

 Trainable params: 22,644,551 (86.38 MB)

 Non-trainable params: 14,717,248 (56.14 MB)

In [9]:
# ---------------- Callbacks ----------------
ckpt_path = f"{OUT_DIR}/best_model.keras"  # use native Keras format
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, mode="max", verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
]

# ---------------- Train ----------------
EPOCHS = 15 # start modest; increase if stable
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/15
875/875 ━━━━━━━━━━━━━━━━━━━━ 0s 220ms/step - accuracy: 0.2882 - loss: 2.4614
Epoch 1: val_accuracy improved from -inf to 0.43872, saving model to /content/drive/MyDrive/Colab Notebooks/Emotion_VGG16_80_20_Improved_Results_ab20251009_135740/best_model.keras
875/875 ━━━━━━━━━━━━━━━━━━━━ 267s 278ms/step - accuracy: 0.2883 - loss: 2.4611 - val_accuracy: 0.4387 - val_loss: 1.6094 - learning_rate: 1.0000e-04
Epoch 2/15
875/875 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step - accuracy: 0.4223 - loss: 1.7916
Epoch 2: val_accuracy improved from 0.43872 to 0.46866, saving model to /content/drive/MyDrive/Colab Notebooks/Emotion_VGG16_80_20_Improved_Results_ab20251009_135740/best_model.keras
875/875 ━━━━━━━━━━━━━━━━━━━━ 228s 261ms/step - accuracy: 0.4223 - loss: 1.7916 - val_accuracy: 0.4687 - val_loss: 1.4740 - learning_rate: 1.0000e-04
Epoch 3/15
875/875 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step - accuracy: 0.4600 - loss: 1.5815
Epoch 3: val_accuracy improved from 0.46866 to 0.48189, saving model to /co

In [10]:
# Save model & history
model.save(os.path.join(OUT_DIR, "hybrid_vgg16_cnn_final.keras"))
pd.DataFrame(history.history).to_csv(os.path.join(OUT_DIR, "training_history.csv"), index=False)


In [11]:
# ---------------- Curves ----------------
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history.history["accuracy"], label="Train Acc")
plt.plot(history.history["val_accuracy"], label="Val Acc")
plt.title("Accuracy Curve"); plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.title("Loss Curve"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=150)
plt.close()
print("Training curves saved.")


Training curves saved.


In [12]:
# ---------------- Evaluate on Test ----------------
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"TEST -> loss: {test_loss:.4f} | acc: {test_acc:.4f}")


TEST -> loss: 1.2617 | acc: 0.5348


In [13]:
# ---------------- Predictions & Reports ----------------
# true labels (one-hot)
y_true = []
for _, y in test_ds.unbatch():
    y_true.append(y.numpy())
y_true = np.array(y_true)
y_true_cls = np.argmax(y_true, axis=1)

# probs & preds
y_prob = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

# Classification Report
rep = classification_report(y_true_cls, y_pred, target_names=class_names, digits=4)
print(rep)
with open(os.path.join(OUT_DIR, "classification_report.txt"), "w") as f:
    f.write(rep)

# Confusion Matrix
cm = confusion_matrix(y_true_cls, y_pred)
plt.figure(figsize=(7.2,6.2))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.title("Confusion Matrix"); plt.colorbar()
ticks = np.arange(num_classes)
plt.xticks(ticks, class_names, rotation=45, ha="right")
plt.yticks(ticks, class_names)
th = cm.max()/2.0
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(j, i, cm[i, j], ha="center", color="white" if cm[i, j] > th else "black", fontsize=8)
plt.ylabel("True"); plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=150)
plt.close()
print("Confusion matrix saved.")

# ---------------- ROC Curves (OvR + micro) ----------------
y_true_bin = y_true  # already one-hot
num_classes = y_true_bin.shape[1]
fpr, tpr, roc_auc = {}, {}, {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

fpr["micro"], tpr["micro"], _ = roc_curve(y_true_bin.ravel(), y_prob.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

plt.figure(figsize=(8,6))
plt.plot(fpr["micro"], tpr["micro"], label=f"micro-average ROC (AUC = {roc_auc['micro']:.3f})", linewidth=2)
colors = ['aqua', 'darkorange', 'cornflowerblue', 'green', 'red', 'purple', 'brown']
for i, cname in enumerate(class_names):
    plt.plot(fpr[i], tpr[i], lw=2, color=colors[i % len(colors)], label=f"{cname} (AUC = {roc_auc[i]:.3f})")
plt.plot([0,1], [0,1], "k--", lw=1)
plt.xlim([0,1]); plt.ylim([0,1.05])
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("Multi-class ROC (One-vs-Rest)")
plt.legend(loc="lower right", fontsize=8)
plt.savefig(os.path.join(OUT_DIR, "roc_curves.png"), dpi=150)
plt.close()
print("ROC curves saved:", os.path.join(OUT_DIR, "roc_curves.png"))

print("✅ All outputs saved to:", OUT_DIR)

              precision    recall  f1-score   support

       angry     0.3986    0.4248    0.4113       958
   disgusted     0.4255    0.3604    0.3902       111
     fearful     0.4270    0.3428    0.3803      1024
       happy     0.7036    0.7441    0.7233      1774
     neutral     0.4729    0.5101    0.4908      1233
         sad     0.4462    0.4387    0.4424      1247
   surprised     0.6737    0.6558    0.6646       831

    accuracy                         0.5348      7178
   macro avg     0.5068    0.4967    0.5004      7178
weighted avg     0.5313    0.5348    0.5320      7178

Confusion matrix saved.
ROC curves saved: /content/drive/MyDrive/Colab Notebooks/Emotion_VGG16_80_20_Improved_Results_ab20251009_135740/roc_curves.png
✅ All outputs saved to: /content/drive/MyDrive/Colab Notebooks/Emotion_VGG16_80_20_Improved_Results_ab20251009_135740
